from nn import *
from ray import train, tune
from ray.tune.search.hyperopt import HyperOptSearch


def Mix(i, rqh=0, rkh=0, rk=0, rvh=0, rv=0, roh=0, cch=0, cc=0, coh=0, ah=0, mh=0, sh=0, h=0, oh=0, j=0, o=0, rh=1, cg=1, rr=1, cr=1, ar=1, mr=1, b=False):
    h = h or i
    o = o or h
    return Add(
        Seq(
            Add(
                Ret(i, rqh, rkh, rk, rvh, rv, roh, h, rh, rr, min=1, max=64),
                SepCovLeft(i, cch, cc, coh, h, cg, cr),
                *[ApLeft(i, ah, h, ar)] if ah else [],
                *[MpLeft(i, mh, h, mr)] if mh else [],
                *[SeLeft(i, sh, h)] if sh else [],
            ),
            Bn(h),
            SiLU(True),
            Sep(h, oh, o),
        ),
        Sep(i, j, o, b),
    )


def Mixs(e, m, **_):
    return Seq(
        Embedding(e, m[0]["i"]),
        *[
            Mix(
                **dict(zip(m, M)),
                o=dict(zip(m, o))["i"] if b or B else e,
                b=not (b or B),
            )
            for b, m, M in pk(m)
            for B, M, o in pk(np.linspace(*[m.pop("l", 1) + 1, [M[m] for m in m], list(m.values())][::-1], dtype=int).tolist())
        ],
    )


def train_mixs(O, lu=lambda x: x, type=torch.bfloat16, compile=lambda m, x: torch.compile(torch.jit.trace(m, x)), tqdm=lambda x: x, log=lambda l, a: train.report({"l": float(l), "a": float(a.mean())})):
    x = Dat("py.txt", 1, 8192, 32)
    m = lu(sized(Mixs, O, 64))
    print(re.sub(r"((.+\n)+?)\1+", r"\1", re.sub("\(\d+\): ", "", str(m))))
    m = compile(m.to("cuda", type), x())
    o = torch.optim.Adam(m.parameters(), **O.get("o", {}))
    O = torch.optim.lr_scheduler.OneCycleLR(o, **dict(max_lr=0.001, total_steps=len(x), pct_start=0.01, div_factor=9, final_div_factor=99) | O.get("O", {}))
    L = A = 0
    for i, x in enumerate(tqdm(x), 1):
        y = m(x[:, :-1])
        x = x[:, 1:]
        l = functional.cross_entropy(y.mT, x)
        l.backward()
        o.step()
        o.zero_grad()
        O.step()
        L += l.data.log().float()
        A += (y.argmax(2) != x).float().mean(0).log()
        if not i % (O.total_steps // 100):
            log((L * 100 / O.total_steps).exp(), 1 - (A * 100 / O.total_steps).exp())
            L = A = 0

m|k|v|h|o|n|acc|los
---|---|---|---|---|---|---|---
64|8|8|8|128|2|37|
128|11|11|11|256|2|33|
256|16|16|16|512|1|36|
256|16|16|16|512|2|30|
256|32|32|16|512|2|28|
256|32|32|8|512|2|29|
256|32|32|4|512|2|30|
256|64|64|4|512|2|28|
256|64|64|2|512|2|28|114
256|16|16|16|512|3|32|
256|1|256|256|512|2|25|
256|1|256|256|512|2|25|
256|256|256|1|512|2|34|136
256|128|128|2|512|2|32|128
256|256|256|2|512|2|32|126

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import Output
from nn import *
from tqdm.notebook import tqdm

In [ ]:
m, h, o, n = 128, 32768 * 2, 512, 2
x = Dat("py.txt", 1, 128, 32)
m = Seq(
    Embedding(68, m),
    *(
        Add(
            Seq(
                Add(
                    Seq(
                        Mul(
                            Lin(m, h, True),
                            Seq(
                                Mul(
                                    Lin(m, h, True),
                                    Lin(m, h, True),
                                ),
                                Decay(h, d=-2, shift=1),
                            ),
                        ),
                        Lin(h, o),
                    ),
                    Lin(m, o),
                ),
                Bn(o),
                SiLU(True),
                Lin(o, m if i else 68),
            ),
            Lin(m, m if i else 68, not i),
        )
        for i in range(n)[::-1]
    ),
)
m = torch.compile(torch.jit.trace(m.to("cuda", torch.bfloat16), x()))
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.003, len(x), pct_start=0.01, div_factor=3, final_div_factor=99)
L = Log(10)
A = Log(10)
out = Output()
display(out)
for i, x in enumerate(tqdm(x), 1):
    y = m(x[:, :-1])
    x = x[:, 1:]
    l = functional.cross_entropy(y.mT, x)
    l.backward()
    o.step()
    o.zero_grad()
    O.step()
    L(l)
    A((y.argmax(2) != x).sum() / x.numel())
    if not i % 1000 or i == O.total_steps:
        plt.plot(L())
        plt.twinx().plot(1 - A())
        out.clear_output(True)
        with out:
            plt.show()
x = L()[-10:].mean()
torch.save(str(m), f"{x:.3f}")
x

In [ ]:
m, h, o, n = 128 * 2, 32768 * 2, 512, 2
x = Dat("py.txt", 1, 128, 32)
m = Seq(
    Embedding(68, m),
    *(
        Add(
            Seq(
                Add(
                    Seq(
                        Mul(
                            Lin(m, h, True),
                            Seq(
                                Mul(
                                    Lin(m, h, True),
                                    Lin(m, h, True),
                                ),
                                Decay(h, d=-2, shift=1),
                            ),
                        ),
                        Lin(h, o),
                    ),
                    Lin(m, o),
                ),
                Bn(o),
                SiLU(True),
                Lin(o, m if i else 68),
            ),
            Lin(m, m if i else 68, not i),
        )
        for i in range(n)[::-1]
    ),
)
m = torch.compile(torch.jit.trace(m.to("cuda", torch.bfloat16), x()))
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.003, len(x), pct_start=0.01, div_factor=3, final_div_factor=99)
L = Log(10)
A = Log(10)
out = Output()
display(out)
for i, x in enumerate(tqdm(x), 1):
    y = m(x[:, :-1])
    x = x[:, 1:]
    l = functional.cross_entropy(y.mT, x)
    l.backward()
    o.step()
    o.zero_grad()
    O.step()
    L(l)
    A((y.argmax(2) != x).sum() / x.numel())
    if not i % 1000 or i == O.total_steps:
        plt.plot(L())
        plt.twinx().plot(1 - A())
        out.clear_output(True)
        with out:
            plt.show()
x = L()[-10:].mean()
torch.save(str(m), f"{x:.3f}")
x

In [ ]:
m, h, o, n = 128, 32768 * 2 * 2, 512, 2
x = Dat("py.txt", 1, 128, 32)
m = Seq(
    Embedding(68, m),
    *(
        Add(
            Seq(
                Add(
                    Seq(
                        Mul(
                            Lin(m, h, True),
                            Seq(
                                Mul(
                                    Lin(m, h, True),
                                    Lin(m, h, True),
                                ),
                                Decay(h, d=-2, shift=1),
                            ),
                        ),
                        Lin(h, o),
                    ),
                    Lin(m, o),
                ),
                Bn(o),
                SiLU(True),
                Lin(o, m if i else 68),
            ),
            Lin(m, m if i else 68, not i),
        )
        for i in range(n)[::-1]
    ),
)
m = torch.compile(torch.jit.trace(m.to("cuda", torch.bfloat16), x()))
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.003, len(x), pct_start=0.01, div_factor=3, final_div_factor=99)
L = Log(10)
A = Log(10)
out = Output()
display(out)
for i, x in enumerate(tqdm(x), 1):
    y = m(x[:, :-1])
    x = x[:, 1:]
    l = functional.cross_entropy(y.mT, x)
    l.backward()
    o.step()
    o.zero_grad()
    O.step()
    L(l)
    A((y.argmax(2) != x).sum() / x.numel())
    if not i % 1000 or i == O.total_steps:
        plt.plot(L())
        plt.twinx().plot(1 - A())
        out.clear_output(True)
        with out:
            plt.show()
x = L()[-10:].mean()
torch.save(str(m), f"{x:.3f}")
x

In [ ]:
m, h, o, n = 128, 32768 * 2, 512 * 2, 2
x = Dat("py.txt", 1, 128, 32)
m = Seq(
    Embedding(68, m),
    *(
        Add(
            Seq(
                Add(
                    Seq(
                        Mul(
                            Lin(m, h, True),
                            Seq(
                                Mul(
                                    Lin(m, h, True),
                                    Lin(m, h, True),
                                ),
                                Decay(h, d=-2, shift=1),
                            ),
                        ),
                        Lin(h, o),
                    ),
                    Lin(m, o),
                ),
                Bn(o),
                SiLU(True),
                Lin(o, m if i else 68),
            ),
            Lin(m, m if i else 68, not i),
        )
        for i in range(n)[::-1]
    ),
)
m = torch.compile(torch.jit.trace(m.to("cuda", torch.bfloat16), x()))
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.003, len(x), pct_start=0.01, div_factor=3, final_div_factor=99)
L = Log(10)
A = Log(10)
out = Output()
display(out)
for i, x in enumerate(tqdm(x), 1):
    y = m(x[:, :-1])
    x = x[:, 1:]
    l = functional.cross_entropy(y.mT, x)
    l.backward()
    o.step()
    o.zero_grad()
    O.step()
    L(l)
    A((y.argmax(2) != x).sum() / x.numel())
    if not i % 1000 or i == O.total_steps:
        plt.plot(L())
        plt.twinx().plot(1 - A())
        out.clear_output(True)
        with out:
            plt.show()
x = L()[-10:].mean()
torch.save(str(m), f"{x:.3f}")
x

def Ret(i, qh=0, kh=0, k=0, vh=0, v=0, oh=0, o=0, h=0, r=1, b=False, **_):
    k = k or i
    v = v or k
    h = h or k
    o = o or i
    return Sequential(
        Dot(
            Sequential(Sep(i, h * qh, h * k, True), Unflatten(-1, [h, 1, k])),
            Sequential(
                Mul(
                    Sequential(Sep(i, h * kh, h * k, True), Unflatten(-1, [h, 1, k])),
                    Sequential(Sep(i, h * vh, h * v, True), Unflatten(-1, [h, v, 1])),
                ),
                Decay((h, 1, k), (h, v, 1), d=-4, shift=1, **_),
            ),
        ),
        Flatten(-3),
        Sep(h * v, h * oh, o),
    )


m, k, v, h, o, n = 64, 4, 4, 4096, 64, 2
x = Dat("py.txt", 1, 128, 32)
m = Seq(
    Embedding(68, m),
    *(
        Add(
            Seq(
                Add(
                    Ret(m, k=k, v=v, h=h, o=o),
                    Lin(m, o),
                ),
                Bn(o),
                SiLU(True),
                Lin(o, m if i else 68),
            ),
            Lin(m, m if i else 68, not i),
        )
        for i in range(n)[::-1]
    ),
)
m = torch.compile(torch.jit.trace(m.to("cuda", torch.bfloat16), x()))
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.003, len(x), pct_start=0.01, div_factor=3, final_div_factor=99)
L = Log(10)
A = Log(10)
out = Output()
display(out)
for i, x in enumerate(tqdm(x), 1):
    y = m(x[:, :-1])
    x = x[:, 1:]
    l = functional.cross_entropy(y.mT, x)
    l.backward()
    o.step()
    o.zero_grad()
    O.step()
    L(l)
    A((y.argmax(2) != x).sum() / x.numel())
    if not i % 1000 or i == O.total_steps:
        plt.plot(L())
        plt.twinx().plot(1 - A())
        out.clear_output(True)
        with out:
            plt.show()
L()[-100:].mean()